# *GAMESS* Workflow

Here will be presented the **step by step** workflow for calculating the energy profile for rotating the central bond of butane. We have seen that butane has energy minima at torsion angles of 180&deg; and 67&deg;. It also has a maximum at 0&deg;. Is the other maximum at 120&deg; or is it slightly **offset** due to the size difference between a hydrogen and a methyl group? We will make many input files with locked torsion angles varying from 0&deg; to 180&deg; and then calculate the **optimized** structures at each locked angle. The energies will be extracted from the output files and we will then plot the potential energy profile for bond rotation.

In the original version of this page each step was a *Unix* shell script or a drag-and-drop into *GamessQ*. Here every step is a cell of *Python* in this notebook. Run the settings cell first.

In [1]:
# ===== SETTINGS =========================================================
# Your GAMESS location is set ONCE, in gamess_settings.py (next to this
# notebook). Edit that file, save it, and restart the kernel.
from pathlib import Path
import gamess_tools as gt

INPUTS = Path("inputs")     # input files that come with the book
RUNS = Path("runs")         # where log files and new input files are written
DATA = Path("data")         # the collected results are saved here
RUNS.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)
# ========================================================================

## The Input File

First we **write** an input file for butane. We will not use symmetry because we want to rotate the bond to many different angles. We will use the *Z-matrix* method because it will identify the central **torsion** as a coordinate. Then we can **lock** that coordinate with `IFREEZ(1)=6` in the `$STATPT` group. The value of the torsion on line 4 of the *Z-matrix* is the placeholder text `NUMBER`. The file is `inputs/butane_SCAN.inp`.

In [2]:
template = INPUTS / "butane_SCAN.inp"
print(template.read_text())

 $CONTRL 
     SCFTYP=RHF 
     RUNTYP=OPTIMIZE 
     MAXIT=30 
     MULT=1 
     COORD=ZMT 
     NZVAR=36 
 $END
 $SYSTEM 
     TIMLIM=525600 
     MEMORY=1000000 
 $END
 $BASIS 
     GBASIS=N21 
     NGAUSS=3 
 $END
 $SCF DIRSCF=.TRUE. $END
 $STATPT 
     OPTTOL=0.0001 
     NSTEP=1000 
     IFREEZ(1)=6 $END
 $DATA 
Butane
C1
C
C   1    1.54000
C   2    1.54000  1 109.0
C   1    1.54000  2 109.4712   3   NUMBER
H   1    1.14000  2 109.4712   4 -120.0000
H   1    1.14000  5 108.5      2  120.0000
H   2    1.14000  1 109.4712   3 -120.0000
H   2    1.14000  7 109.4712   1  120.0000
H   3    1.14000  2 109.4712   8    0.0000
H   3    1.14000  9 109.4712   2  120.0000
H   3    1.14000  9 109.4712  10  120.0000
H   4    1.14000  1 109.0      6   -0.0000
H   4    1.14000  12 108.5     1  120.0000
H   4    1.14000  12 109.4712 13  119.0000
 $END
 $ZMAT IZMAT(1)=1,2,1, 
                1,3,2,  2,3,2,1, 
                1,4,1,  2,4,1,2,   3,4,1,2,3, 
                1,5,1,  2,5,1,2,   3,5,1,2

You will see that not all angles are 109.47&deg;. I **jiggled** some to avoid any perfect symmetry locks that prevented the energy minimization from converging in one of the jobs last time.

## Making the Series

The original book used a short **shell script** that ran the *sed* tool to replace the string `NUMBER` with each number from a list and write a new file with a name based on that number:

```bash
input_file=butane-SCAN.inp
spacer=_
for num in $(seq -w 000 010 180)
do
  sed "s/NUMBER/$num/" $input_file > "$input_file$spacer$num.inp"
done
```

The *Python* version below does the same thing. `range(0, 181, 10)` makes the list 0, 10, 20 … 180 (like `seq 000 010 180`), and `f"{angle:03d}"` writes each number with three digits (000, 010, …), like the `-w` option of *seq*. `gt.substitute()` is our *Python* stand-in for *sed*: it copies the template and swaps `NUMBER` for the angle.

In [3]:
inputs = []
for angle in range(0, 181, 10):
    new_file = RUNS / f"butane_SCAN_{angle:03d}.inp"
    gt.substitute(template, new_file, {"NUMBER": f"{angle:03d}"})
    inputs.append(new_file)

print(len(inputs), "input files written:")
print([f.name for f in inputs])

19 input files written:
['butane_SCAN_000.inp', 'butane_SCAN_010.inp', 'butane_SCAN_020.inp', 'butane_SCAN_030.inp', 'butane_SCAN_040.inp', 'butane_SCAN_050.inp', 'butane_SCAN_060.inp', 'butane_SCAN_070.inp', 'butane_SCAN_080.inp', 'butane_SCAN_090.inp', 'butane_SCAN_100.inp', 'butane_SCAN_110.inp', 'butane_SCAN_120.inp', 'butane_SCAN_130.inp', 'butane_SCAN_140.inp', 'butane_SCAN_150.inp', 'butane_SCAN_160.inp', 'butane_SCAN_170.inp', 'butane_SCAN_180.inp']


Nineteen new files are created with angles for the `4,1,2,3` torsion varying from 0 to 180 in steps of 10. Open any of them in a text editor and check that line 4 of the *Z-matrix* now holds a number.

In [4]:
for line in gt.grep("C   1    1.54000  2 109.4712   3", RUNS / "butane_SCAN_120.inp"):
    print(line)

C   1    1.54000  2 109.4712   3   120


## Running the Jobs

We now have nineteen jobs to submit to *GAMESS*. The original book dropped the files into the *GamessQ* window and let it run them one after the other. A `for` loop does the same job here. Each optimization takes several seconds on a laptop, so the whole scan takes a few minutes.

`rerun=False` tells `gt.run_gamess()` to skip any job that already has a finished log file in `runs/`. If the loop is interrupted, just run the cell again and it picks up where it stopped. Delete the log files (or use `rerun=True`) to start over.

In [5]:
logs = [gt.run_gamess(inp, rerun=False) for inp in inputs]

butane_SCAN_000: using existing butane_SCAN_000.log (rerun=True to run again)
butane_SCAN_010: using existing butane_SCAN_010.log (rerun=True to run again)
butane_SCAN_020: using existing butane_SCAN_020.log (rerun=True to run again)
butane_SCAN_030: using existing butane_SCAN_030.log (rerun=True to run again)
butane_SCAN_040: using existing butane_SCAN_040.log (rerun=True to run again)
butane_SCAN_050: using existing butane_SCAN_050.log (rerun=True to run again)
butane_SCAN_060: using existing butane_SCAN_060.log (rerun=True to run again)
butane_SCAN_070: using existing butane_SCAN_070.log (rerun=True to run again)
butane_SCAN_080: using existing butane_SCAN_080.log (rerun=True to run again)
butane_SCAN_090: using existing butane_SCAN_090.log (rerun=True to run again)
butane_SCAN_100: using existing butane_SCAN_100.log (rerun=True to run again)
butane_SCAN_110: using existing butane_SCAN_110.log (rerun=True to run again)
butane_SCAN_120: using existing butane_SCAN_120.log (rerun=True 

## Extracting the Results

When the jobs are all complete we **extract** the last `NSERCH` line (the one with the energy and the gradient) from each file and save this set of lines in a single result file that we will name `butane_result.txt`. The original book did this with another shell script:

```bash
for name in butane_SCAN*.log; do
    grep -H "NSERCH" "$name" | grep "GRAD" | tail -n 1
done
```

and the author wrote: *"Everyday I promise myself that I will rewrite them in Python."* Here it is. `gt.grep()` returns the lines that contain some text, `[-1]` takes the last of them (like `tail -n 1`), and putting the file name in front of the line copies the `-H` option of *grep*.

In [6]:
result_lines = []
for log in logs:
    lines = [line for line in gt.grep("NSERCH", log) if "GRAD" in line]
    result_lines.append(f"{log.name}: {lines[-1]}")

result_file = DATA / "butane_result.txt"
result_file.write_text("\n".join(result_lines) + "\n")
print("wrote", result_file)

wrote data/butane_result.txt


## Inspecting the Result

If you open the `data/butane_result.txt` file in a text editor you will see nineteen lines of data, one line for each file.

In [7]:
print(result_file.read_text())

butane_SCAN_000.log:  NSERCH:  22  E=     -156.4228465105  GRAD. MAX=  0.0000535  R.M.S.=  0.0000171
butane_SCAN_010.log:  NSERCH:  18  E=     -156.4233726954  GRAD. MAX=  0.0000801  R.M.S.=  0.0000247
butane_SCAN_020.log:  NSERCH:  16  E=     -156.4247889427  GRAD. MAX=  0.0000743  R.M.S.=  0.0000226
butane_SCAN_030.log:  NSERCH:  16  E=     -156.4267021324  GRAD. MAX=  0.0000576  R.M.S.=  0.0000202
butane_SCAN_040.log:  NSERCH:  19  E=     -156.4286427109  GRAD. MAX=  0.0000494  R.M.S.=  0.0000153
butane_SCAN_050.log:  NSERCH:  30  E=     -156.4301846915  GRAD. MAX=  0.0000774  R.M.S.=  0.0000194
butane_SCAN_060.log:  NSERCH:  22  E=     -156.4310610865  GRAD. MAX=  0.0000787  R.M.S.=  0.0000309
butane_SCAN_070.log:  NSERCH:  29  E=     -156.4312168554  GRAD. MAX=  0.0000616  R.M.S.=  0.0000227
butane_SCAN_080.log:  NSERCH:  29  E=     -156.4306888896  GRAD. MAX=  0.0000614  R.M.S.=  0.0000249
butane_SCAN_090.log:  NSERCH:  27  E=     -156.4295738742  GRAD. MAX=  0.0000884  R.M.S.=  

Look at the `NSERCH` column, the number of steps each optimization needed. When the author ran this scan with an earlier version of *GAMESS*, three of the jobs (100&deg;, 130&deg; and 160&deg;) took a **longer time** to converge: 390, 362 and 646 steps. With *GAMESS* 2023 R2 those three finished in about 30 steps, but now the 120&deg; job needed 564 steps. The input file is the same; small differences in the program are enough to send an optimization down a different path.

When a job does take hundreds of steps, we could open it in *MacMolPlt* and watch a "movie" of its journey and plot the energies along the way. Where did it get stuck? My slow jobs all eventually found their way home, but only because this was a light-weight theory level and I was confident in **allowing** for a large number of steps (`NSTEP=1000`). If the level of theory was higher I might have set the **limit** to 100 steps, or even 30 steps.

Why would a job take so long? **Bad luck** with the exact structure of the input starts it on the long path. Tweak a few bond lengths or angles and it might converge just like the others. Clearly, **how you start matters**. For high levels of theory it is important to start as close to the final answer as possible.

We can check the gradient for every file at once. Every job should have stopped because its largest gradient fell below `OPTTOL=0.0001`, and not because it ran out of steps.

In [8]:
for log in logs:
    step, energy, grad_max, rms = gt.optimization_steps(log)[-1]
    status = "converged" if grad_max < 0.0001 else "NOT CONVERGED"
    print(f"{log.name:22s} steps = {step:4d}   E = {energy:.10f}   {status}")

butane_SCAN_000.log    steps =   22   E = -156.4228465105   converged
butane_SCAN_010.log    steps =   18   E = -156.4233726954   converged
butane_SCAN_020.log    steps =   16   E = -156.4247889427   converged
butane_SCAN_030.log    steps =   16   E = -156.4267021324   converged
butane_SCAN_040.log    steps =   19   E = -156.4286427109   converged
butane_SCAN_050.log    steps =   30   E = -156.4301846915   converged
butane_SCAN_060.log    steps =   22   E = -156.4310610865   converged
butane_SCAN_070.log    steps =   29   E = -156.4312168554   converged
butane_SCAN_080.log    steps =   29   E = -156.4306888896   converged
butane_SCAN_090.log    steps =   27   E = -156.4295738742   converged
butane_SCAN_100.log    steps =   30   E = -156.4282209524   converged
butane_SCAN_110.log    steps =   27   E = -156.4271414730   converged
butane_SCAN_120.log    steps =  564   E = -156.4267262798   converged
butane_SCAN_130.log    steps =   25   E = -156.4271135021   converged
butane_SCAN_140.log 

```{note}
**The number of processors matters too.** This check is not a formality. When these jobs were run again with *GAMESS* 2024 R2, the same input files gave different paths when only the number of processors changed. For example, the 120&deg; job needed 801, 950, 240 and 597 steps on 1, 2, 4 and 8 processors. All of these runs converged. In the main chapter, the 130&deg; job with the original (un-jiggled) input stalled at the 1000-step limit in two of seven runs on 4 processors, yet *GAMESS* still reported that it had terminated normally. A parallel run adds up its pieces in a slightly different order each time, and on a long, flat path that tiny difference can be enough to change the outcome. Only a one-processor run repeats exactly. See [](#processors-convergence) for the details. Always check that every optimization actually converged, rather than trusting the "terminated normally" message.
```

We now have the raw data. Next we will process it and **plot the results** using *Python*. The [next subchapter](5_Rotation_Profiles_Workflow_Data.ipynb) is a notebook that details this workflow.

## AI Statement

This page was converted from the original Markdown page `5_Rotation_Profiles_Workflow.md` to a *Jupyter* notebook with the help of Claude (Anthropic), following the author's instructions.

- **Original text kept (about 60 %).** The introduction, the description of the input file, and the discussion of slow-converging jobs and of how the starting structure matters are the author's words; typos were fixed. The two shell scripts are kept for comparison.
- **Changed workflow.** The *sed* script, *GamessQ* and the *grep* script are replaced by *Python* cells (`gt.substitute()`, `gt.run_gamess()` and `gt.grep()`), so the three terminal and *GamessQ* screenshots were removed. The result file is now `data/butane_result.txt` and its file names no longer carry a *GamessQ* job number. The input file is now named `butane_SCAN.inp` to match its log files.
- **Written by AI.** The settings cell, all code cells and the paragraphs that explain them, the convergence check, the note that with *GAMESS* 2023 R2 a different job (120&deg;) was the slow one, and the note on how the number of processors changed the step counts with *GAMESS* 2024 R2.